# Predicting F1 Pit Stops

Link to Competition: https://www.kaggle.com/competitions/playground-series-s6e5/overview

**Metric:** ROC-AUC (rank-based, threshold-independent — submit probabilities, not 0/1)

## Imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.figsize'] = (12, 6)

import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

import xgboost as xgb
import optuna

from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, mean_squared_error, classification_report, balanced_accuracy_score
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold, StratifiedKFold, StratifiedGroupKFold, GroupKFold, RandomizedSearchCV, train_test_split
from sklearn.cluster import KMeans
from sklearn.utils.class_weight import compute_sample_weight

from common import *

In [2]:
from platform import python_version
print('python: ', python_version())
print('pandas: ', pd.__version__)
print('numpy: ', np.__version__)
import matplotlib
print('matplotlib: ', matplotlib.__version__)
print('seaborn: ', sns.__version__)
import sklearn
print('sklearn: ', sklearn.__version__)
print('xgboost: ', xgb.__version__)

python:  3.13.11
pandas:  2.3.3
numpy:  2.3.5
matplotlib:  3.10.8
seaborn:  0.13.2
sklearn:  1.8.0
xgboost:  3.2.0


## Helpers

## Load data

In [3]:
train_df = pd.read_csv('archive/train.csv')
test_df = pd.read_csv('archive/test.csv')
orig_df = pd.read_csv('archive/f1_strategy_dataset_v4.csv')

## Call the pipeline

In [4]:
df = (train_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

## Concat original dataset

In [5]:
orig_df_cleaned = (orig_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

In [6]:
train_df_length = df.shape[0]

In [7]:
orig_df_cleaned_length = orig_df_cleaned.shape[0]

In [8]:
df = pd.concat([df, orig_df_cleaned])

In [9]:
if 'normalized_tyrelife' in df.columns:
    df = df.drop('normalized_tyrelife', axis=1)

In [10]:
df = df.reset_index(drop=True)

In [11]:
# confirm that I know where the boundaries are
a = df.iloc[train_df_length]
b = orig_df_cleaned.iloc[0].drop('normalized_tyrelife')
a.sort_index().equals(b.sort_index())

True

In [12]:
sample_weights = np.ones(df.shape[0])

In [13]:
sample_weights[train_df_length:] = 1.25

## Features

In [14]:
target = get_target()

In [15]:
features = get_features(df)

In [16]:
features

['compound',
 'race',
 'year',
 'pitstop',
 'lapnumber',
 'stint',
 'tyrelife',
 'position',
 'laptime_(s)',
 'laptime_delta',
 'cumulative_degradation',
 'raceprogress',
 'position_change',
 'year_cat']

In [17]:
categorical_features = [
    # 'driver',
    'compound',
    'race',
    'year_cat'
]

In [18]:
numerical_features = [f for f in features if f not in categorical_features]

In [19]:
categorical_features

['compound', 'race', 'year_cat']

In [20]:
numerical_features

['year',
 'pitstop',
 'lapnumber',
 'stint',
 'tyrelife',
 'position',
 'laptime_(s)',
 'laptime_delta',
 'cumulative_degradation',
 'raceprogress',
 'position_change']

In [21]:
df[features]

,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_(s),laptime_delta,cumulative_degradation,raceprogress,position_change,year_cat
0,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,2022
1,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,2025
2,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,2022
3,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,2023
4,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540506,HARD,United States Grand Prix,2022,0,52,3,29.0,12,103.373,26.850,-90.264,0.722222,2.0,2022
540507,HARD,United States Grand Prix,2022,0,53,3,30.0,13,105.200,27.061,-88.437,0.736111,1.0,2022
540508,HARD,United States Grand Prix,2022,0,54,3,31.0,13,104.102,27.149,-89.535,0.750000,0.0,2022
540509,HARD,United States Grand Prix,2022,0,55,3,32.0,14,103.812,4.428,-89.825,0.763889,-1.0,2022


In [22]:
df[categorical_features] = df[categorical_features].astype('category')

In [23]:
categorical_features

['compound', 'race', 'year_cat']

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 540511 entries, 0 to 540510
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype   
---  ------                  --------------   -----   
 0   compound                540445 non-null  category
 1   race                    540511 non-null  category
 2   year                    540511 non-null  int64   
 3   pitstop                 540511 non-null  int64   
 4   lapnumber               540511 non-null  int64   
 5   stint                   540511 non-null  int64   
 6   tyrelife                540511 non-null  float64 
 7   position                540511 non-null  int64   
 8   laptime_(s)             540511 non-null  float64 
 9   laptime_delta           540511 non-null  float64 
 10  cumulative_degradation  540511 non-null  float64 
 11  raceprogress            540511 non-null  float64 
 12  position_change         540511 non-null  float64 
 13  pitnextlap              540511 non-null  float64 
 14  year

## StratifiedGroupKFold Loop

Group by `(race, year)` so the same race never appears in both train and val — keeps CV from leaking race-specific context. Stratification preserves class balance per fold.

In [25]:
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'enable_categorical': True,
    'n_estimators': 2000,
    'early_stopping_rounds': 50,
    'n_jobs': -1,
    'random_state': 123,
    'learning_rate': 0.017826115658430553, 
    'max_depth': 12, 
    'min_child_weight': 3.5110400956253396, 
    'subsample': 0.8552351876274983, 
    'colsample_bytree': 0.5203491325725986, 
    'colsample_bylevel': 0.636439161771222, 
    'gamma': 2.4359346535197697e-07, 
    'reg_alpha': 0.00217038677275064, 
    'reg_lambda': 1.3555364763007913e-06, 
    'max_bin': 256
}

In [26]:
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=123)
scores = []
best_iters = []
oof_preds = np.zeros(len(df))

X, y = df[features], df[target]
groups = (df['race'].astype(str) + '_' + df['year'].astype(str)).values

strat_y = (df['pitnextlap'].astype(str) + '_' + df['year'].astype(str))

In [27]:
# diagnosing StratifiedGroupKFold strategy
# for fold, (_, val_idx) in enumerate(sgkf.split(X, strat_y, groups=groups)):
#     fold_years = df.iloc[val_idx]['year'].value_counts(normalize=True)
#     print(f"Fold {fold}: {dict(fold_years.round(3))}")

In [28]:
for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, strat_y, groups=groups)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBClassifier(**xgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
        sample_weight=sample_weights[train_idx]
    )

    proba = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = proba
    score = roc_auc_score(y_val, proba)
    scores.append(score)
    best_iters.append(model.best_iteration)
    print(f"Fold {fold}: AUC={score:.5f}  best_iter={model.best_iteration} sample_weights_sum={sample_weights[train_idx].sum()}")

print(f"\nMean AUC: {np.mean(scores):.5f} ± {np.std(scores):.5f}")
print(f"Mean best_iter: {int(np.mean(best_iters))}")

# Save OOF predictions for blending (train portion only — LB-relevant rows)
oof_train = oof_preds[:train_df_length]
y_train_arr = y.iloc[:train_df_length].values
pd.DataFrame({'oof': oof_train, 'y': y_train_arr}).to_csv('./archive/xgb_oof.csv', index=False)
print(f"\nOOF AUC (train portion): {roc_auc_score(y_train_arr, oof_train):.5f}")
print('saved ./archive/xgb_oof.csv')

Fold 0: AUC=0.91933  best_iter=792 sample_weights_sum=513214.75
Fold 1: AUC=0.92250  best_iter=454 sample_weights_sum=507762.5
Fold 2: AUC=0.94242  best_iter=644 sample_weights_sum=514215.75
Fold 3: AUC=0.92449  best_iter=801 sample_weights_sum=510762.5
Fold 4: AUC=0.94330  best_iter=648 sample_weights_sum=507254.75
Fold 5: AUC=0.94161  best_iter=1112 sample_weights_sum=511685.25
Fold 6: AUC=0.93752  best_iter=1256 sample_weights_sum=505234.0
Fold 7: AUC=0.93108  best_iter=1317 sample_weights_sum=511696.75
Fold 8: AUC=0.93081  best_iter=1226 sample_weights_sum=506126.5
Fold 9: AUC=0.92333  best_iter=1090 sample_weights_sum=504731.0

Mean AUC: 0.93164 ± 0.00862
Mean best_iter: 934

OOF AUC (train portion): 0.93514
saved ./archive/xgb_oof.csv


## Final Model

In [29]:
final_params = {k: v for k, v in xgb_params.items() if k != 'early_stopping_rounds'}
final_params['n_estimators'] = int(np.max(best_iters) * 1.10)

xgb_final_model = XGBClassifier(**final_params)
xgb_final_model.fit(X, y, sample_weight=sample_weights)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,0.636439161771222
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.5203491325725986
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.data

In [30]:
 pd.Series(xgb_final_model.feature_importances_, index=features).sort_values(ascending=False)

stint                     0.271395
year_cat                  0.161384
year                      0.148145
tyrelife                  0.063516
compound                  0.059489
race                      0.059307
pitstop                   0.048769
raceprogress              0.033063
lapnumber                 0.032564
laptime_delta             0.031193
cumulative_degradation    0.025712
position_change           0.025120
laptime_(s)               0.020615
position                  0.019730
dtype: float32

In [31]:
df = (test_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

In [32]:
df[categorical_features] = df[categorical_features].astype('category')

In [33]:
df

,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_(s),laptime_delta,cumulative_degradation,raceprogress,position_change,year_cat
0,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0,2023
1,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0,2023
2,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0,2023
3,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0,2024
4,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188160,MEDIUM,Australian Grand Prix,2024,1,14,1,14.0,4,83.879,-16.919,-87.767,0.179487,-2.0,2024
188161,SOFT,Pre-Season Testing,2025,0,60,3,26.0,4,95.727,7.920,-36.485,0.789474,-3.0,2025
188162,MEDIUM,Hungarian Grand Prix,2022,0,28,2,21.0,7,85.058,-14.180,-0.339,0.388889,3.0,2022
188163,MEDIUM,Monaco Grand Prix,2024,0,20,2,15.0,7,80.074,-19.004,-37.967,0.256410,0.0,2024


In [34]:
preds = xgb_final_model.predict_proba(df[features])[:, 1]

In [35]:
preds

array([0.00463284, 0.00643668, 0.00338118, ..., 0.76896286, 0.91071117,
       0.01275982], shape=(188165,), dtype=float32)

In [36]:
submission_df = pd.DataFrame(test_df['id'].copy())

In [37]:
submission_df['PitNextLap'] = preds

In [38]:
submission_df

,id,PitNextLap
0,439140,0.004633
1,439141,0.006437
2,439142,0.003381
3,439143,0.170840
4,439144,0.932639
...,...,...
188160,627300,0.008983
188161,627301,0.481966
188162,627302,0.768963
188163,627303,0.910711


In [39]:
submission_df['PitNextLap'].describe()

count    188165.000000
mean          0.196161
std           0.308109
min           0.000020
25%           0.001893
50%           0.014659
75%           0.291377
max           0.997378
Name: PitNextLap, dtype: float64

In [40]:
pd.DataFrame({'id': test_df['id'].values, 'pred': preds}).to_csv('./archive/xgb_test.csv', index=False)
print('saved ./archive/xgb_test.csv')

saved ./archive/xgb_test.csv
